In [26]:
!pip install scipy
import os
import math
from collections import Counter
from scipy import stats


   ---------------------------------------- 0.0/37.1 MB ? eta -:--:--
   - -------------------------------------- 1.6/37.1 MB 9.7 MB/s eta 0:00:04
   --- ------------------------------------ 3.7/37.1 MB 8.8 MB/s eta 0:00:04
   ------ --------------------------------- 5.8/37.1 MB 8.9 MB/s eta 0:00:04
   -------- ------------------------------- 8.1/37.1 MB 9.3 MB/s eta 0:00:04
   ---------- ----------------------------- 10.0/37.1 MB 9.0 MB/s eta 0:00:04
   ------------- -------------------------- 12.3/37.1 MB 9.2 MB/s eta 0:00:03
   ------------------- -------------------- 18.1/37.1 MB 11.3 MB/s eta 0:00:02
   -------------------------- ------------- 24.1/37.1 MB 12.7 MB/s eta 0:00:02
   ------------------------------- -------- 29.1/37.1 MB 13.8 MB/s eta 0:00:01
   ----------------------------------- ---- 33.3/37.1 MB 14.2 MB/s eta 0:00:01
   -------------------------------------- - 35.7/37.1 MB 13.9 MB/s eta 0:00:01
   ---------------------------------------  37.0/37.1 MB 13.6 MB/s eta 


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
def shannon_entropy(data):
    counts = Counter(data)
    entropy = 0
    for count in counts.values():
        p = count / len(data)
        entropy -= p * math.log2(p)
    return entropy

In [28]:
def chi_square_test(data):
    counts = Counter(data)
    expected = len(data) / 256
    chi2 = 0
    for i in range(256):
        observed = counts.get(i, 0)
        chi2 += (observed - expected) ** 2 / expected
    return chi2

In [29]:
def autocorrelation(data, lag=1):
    n = len(data)
    mean = sum(data) / n

    num = 0
    den = 0

    for i in range(n - lag):
        num += (data[i] - mean) * (data[i + lag] - mean)

    for i in range(n):
        den += (data[i] - mean) ** 2

    return num / den

In [30]:
def ks_test(data):
    normalized = [x / 255 for x in data]
    return stats.kstest(normalized, 'uniform')

In [31]:
import sys
sys.path.append("..")

from generators.mt19937 import mt19937
from generators.os_random import os_random

In [32]:
def generate_mt_bytes(n, seed=1234):
    gen = mt19937(seed)
    return bytes(next(gen) & 0xFF for _ in range(n))

In [36]:
def generate_os_bytes(n):
    return os.urandom(n)

In [33]:
N = 1_000_000  # 1 million d’octets

In [34]:
print("=== MT19937 ===")
mt_data = generate_mt_bytes(N)

print("Entropie:", shannon_entropy(mt_data))
print("Chi²:", chi_square_test(mt_data))
print("Autocorr:", autocorrelation(mt_data))
print("KS:", ks_test(mt_data))

=== MT19937 ===
Entropie: 7.999823078361404
Chi²: 245.07750399999986
Autocorr: -0.00032643007322496396
KS: KstestResult(statistic=np.float64(0.00397943137254908), pvalue=np.float64(3.5072526108159105e-14), statistic_location=np.float64(0.996078431372549), statistic_sign=np.int8(-1))


In [37]:
print("\n=== os.urandom ===")
os_data = generate_os_bytes(N)

print("Entropie:", shannon_entropy(os_data))
print("Chi²:", chi_square_test(os_data))
print("Autocorr:", autocorrelation(os_data))
print("KS:", ks_test(os_data))


=== os.urandom ===
Entropie: 7.999795589262509
Chi²: 283.03360000000004
Autocorr: -0.0007920266627449942
KS: KstestResult(statistic=np.float64(0.004226450980392205), pvalue=np.float64(6.084723965961657e-16), statistic_location=np.float64(0.9686274509803922), statistic_sign=np.int8(-1))
